# Random Forest Attendance Model

ランダムフォレストで入場者数を予測し、特徴量重要度を確認する。


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "requirements.txt").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv(RAW_DATA_DIR / "J1_tokyo_home_2015-2024.csv")


In [ ]:
df["コロナ禍ダミー"] = df["シーズン"].apply(lambda x: 1 if x in [2020, 2021] else 0)


In [ ]:
df["国立フラグ"] = df["スタジアム"].str.contains("国立").astype(int)


In [ ]:
import re

# 試合日カラムをstr型に変換し､曜日を抽出
df["試合日"] = df["試合日"].astype(str)

import re

df["曜日"] = df["試合日"].str.extract(r"[（(]([^)）]+)[)）]")
# 土日祝に該当するかどうかを判定
df["休日フラグ"] = df["曜日"].str.contains("土|日|祝").astype(int)
df


In [ ]:
df_weather = pd.read_csv(RAW_DATA_DIR / "weather_tokyo_2015-2024.csv", header=4)
df_weather


In [ ]:
# df_weatherの0列目､1列目､4列目を抽出し､列目をdate, temperature,rainに変更
df_weather = df_weather.iloc[:, [0, 1, 4]]
df_weather.columns = ["date", "temperature", "rain"]
df_weather


In [ ]:
# df_weatherのdateをdatetime型に変換
df_weather["date"] = pd.to_datetime(df_weather["date"], format="%Y/%m/%d")
df_weather


In [ ]:
df["date"] = df["試合日"].str.extract(r"(\d{2}/\d{2}/\d{2})")[0]
df["date"] = pd.to_datetime(df["date"], format="%y/%m/%d")


In [ ]:
# dateをキーにしてdfのマージ
df = pd.merge(df, df_weather, on="date", how="left")
df


In [ ]:
# 時系列特徴量
df["rolling_mean_3"] = df["入場者数"].rolling(window=3).mean()


In [ ]:
# フィルタリング
df_base = df[
    (df["コロナ禍ダミー"] == 0)
    & (df["国立フラグ"] == 0)
    & (df["スタジアム"] == "味スタ")
]


In [ ]:
# ランダムフォレスト
X = pd.concat(
    [
        pd.get_dummies(df_base["アウェイ"], drop_first=True).astype(float),
        df_base[["休日フラグ", "temperature", "rain", "rolling_mean_3"]].astype(float),
    ],
    axis=1,
)
y = df_base["入場者数"].astype(float)


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 学習データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# モデルの学習
model = RandomForestRegressor(n_estimators=100, max_features="log2", random_state=42)
model.fit(X_train, y_train)
# 予測
y_pred = model.predict(X_test)
# 評価 r2_score
from sklearn.metrics import r2_score

print("R2:", r2_score(y_test, y_pred))


In [ ]:
# feature importance
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame(
    {"feature": feature_names, "importance": importances}
)
feature_importance_df = feature_importance_df.sort_values(
    by="importance", ascending=False
)
plt.rcParams["font.family"] = "Hiragino Sans"
plt.figure(figsize=(10, 6))
sns.barplot(x="importance", y="feature", data=feature_importance_df)
plt.title("Feature Importance")
plt.show()


In [ ]:
# フィルタリングなし
# ランダムフォレスト
X = pd.concat(
    [
        pd.get_dummies(df["アウェイ"], drop_first=True).astype(float),
        df[
            [
                "休日フラグ",
                "コロナ禍ダミー",
                "国立フラグ",
                "temperature",
                "rain",
                "rolling_mean_3",
            ]
        ].astype(float),
    ],
    axis=1,
)
y = df["入場者数"].astype(float)

from re import A
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 学習データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# モデルの学習
model = RandomForestRegressor(n_estimators=100, max_features="log2", random_state=42)
model.fit(X_train, y_train)
# 予測
y_pred = model.predict(X_test)
# 評価 r2_score
from sklearn.metrics import r2_score

print("R2:", r2_score(y_test, y_pred))


In [ ]:
# SHAP図
import shap

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test, check_additivity=False)
shap.plots.beeswarm(shap_values, max_display=10)


In [ ]:
# コロナ禍除く
df_noncovid = df[df["コロナ禍ダミー"] == 0]
# ランダムフォレスト
X = pd.concat(
    [
        pd.get_dummies(df_noncovid["アウェイ"], drop_first=True).astype(float),
        df_noncovid[
            ["休日フラグ", "国立フラグ", "temperature", "rain", "rolling_mean_3"]
        ].astype(float),
    ],
    axis=1,
)
y = df_noncovid["入場者数"].astype(float)

from re import A
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 学習データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# モデルの学習
model = RandomForestRegressor(n_estimators=100, max_features="log2", random_state=42)
model.fit(X_train, y_train)
# 予測
y_pred = model.predict(X_test)
# 評価 r2_score
from sklearn.metrics import r2_score

print("R2:", r2_score(y_test, y_pred))


In [ ]:
# feature importance
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame(
    {"feature": feature_names, "importance": importances}
)
feature_importance_df = feature_importance_df.sort_values(
    by="importance", ascending=False
)
plt.rcParams["font.family"] = "Hiragino Sans"
plt.figure(figsize=(10, 6))
sns.barplot(x="importance", y="feature", data=feature_importance_df)
plt.title("Feature Importance")
plt.show()


In [ ]:
# feature importance
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame(
    {"feature": feature_names, "importance": importances}
)
feature_importance_df = feature_importance_df.sort_values(
    by="importance", ascending=False
)
plt.rcParams["font.family"] = "Hiragino Sans"
plt.figure(figsize=(10, 6))
sns.barplot(x="importance", y="feature", data=feature_importance_df)
plt.title("Feature Importance")
plt.show()


In [ ]:
from sklearn.inspection import permutation_importance

# モデルを学習済みとする
r = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)

# 結果を表示
importance_df = pd.DataFrame(
    {
        "feature": X.columns,
        "importance_mean": r.importances_mean,
        "importance_std": r.importances_std,
    }
).sort_values("importance_mean", ascending=False)

print(importance_df.head(10))


In [ ]:
# permutation importanceの可視化
plt.rcParams["font.family"] = "Hiragino Sans"
plt.figure(figsize=(10, 6))
sns.barplot(x="importance_mean", y="feature", data=importance_df)
plt.title("Permutation Importance")
plt.show()
